# 07 — Models M3 → M6

**Day 5 Track A.** The substantive heart of the paper.

## Plan §6 mapping

| Model | Specification | Tests |
| --- | --- | --- |
| M3a | M2 + Mundlak (`exposure_within` + `exposure_between`), with round dummies | H1, H2, H2a (Mundlak Wald) |
| M3b | M3a without round dummies (γ_W absorbs global trend) | comparison with M3a |
| M4  | M3a + random slope on `genai_i` (group-mean centred) | heterogeneity across countries |
| M5  | M4 + cross-level interaction `exposure_within × eisced` | H4a / H4b / H4₀ |
| M6  | M5 + `epl_c × exposure_within` | H5 institutional moderation |


In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mla.models import (  # noqa: E402
    add_country_year_key,
    build_trust_composite,
    fit_3level,
    master_table,
    mundlak_wald,
    proportional_variance_reduction,
    variance_components,
)

ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"
INTERIM_DIR  = REPO_ROOT / "data" / "interim"
REPO_ROOT

PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA')

## 1. Load + prep — same recipe as notebook 06

In [2]:
def recode_sentinels(s, lo, hi):
    return s.where(s.between(lo, hi))

df = pd.read_parquet(ANALYSIS_DIR / "analysis.parquet")
df = build_trust_composite(df)
df = add_country_year_key(df)
df["agea"]    = recode_sentinels(df["agea"], 14, 110)
df["gndr"]    = recode_sentinels(df["gndr"], 1, 2)
df["eisced"]  = recode_sentinels(df["eisced"], 0, 7)
df["hinctnta"] = recode_sentinels(df["hinctnta"], 1, 10)
df["mnactic"] = recode_sentinels(df["mnactic"], 1, 9)
df["domicil"] = recode_sentinels(df["domicil"], 1, 5)
df["agea_c"] = df["agea"] - 45
df["agea_c_sq"] = df["agea_c"] ** 2
df["female"] = (df["gndr"] == 2).astype("float64")
for _c in ("essround", "isco08", "year"):
    if str(df[_c].dtype).startswith("Int"):
        df[_c] = df[_c].astype("float64")

# genai_i z-standardised within sample (per plan §5).
df["genai_z"] = (df["genai_i"] - df["genai_i"].mean()) / df["genai_i"].std()
# Group-mean-centre the L1 exposure for the random slope and interactions.
g = df.groupby("cntry", observed=True)["genai_z"].transform("mean")
df["genai_z_gmc"] = df["genai_z"] - g

# Centre eisced too so the cross-level interaction has interpretable main effect.
df["eisced_c"] = df["eisced"] - 4  # ISCED 4 = upper-secondary, sample median

df.shape

(276491, 44)

## 2. Fit M2 (baseline) and M3a / M3b

Re-fit M2 on the same sample as M3a so the comparison is on identical N.

In [3]:
REQUIRED = [
    "trust", "genai_i", "genai_z", "genai_z_gmc", "eisced", "eisced_c",
    "agea_c", "female", "mnactic", "domicil", "hinctnta",
    "gdp_growth", "unemp_rate", "hicp_inflation",
    "exposure_ct", "exposure_ct_within", "exposure_ct_between",
]
df_fit = df.dropna(subset=REQUIRED).copy()
print(f"common-sample N (M2..M5): {len(df_fit):,}, {df_fit.cntry.nunique()} countries")

M1_TERMS = (
    "genai_z + C(eisced) + agea_c + agea_c_sq + female "
    "+ C(mnactic) + C(domicil) + hinctnta"
)
M2_TERMS = M1_TERMS + " + gdp_growth + unemp_rate + hicp_inflation + C(essround)"
M3A_TERMS = M2_TERMS + " + exposure_ct_within + exposure_ct_between"
M3B_TERMS = (
    M1_TERMS
    + " + gdp_growth + unemp_rate + hicp_inflation"
    + " + exposure_ct_within + exposure_ct_between"
)

res2  = fit_3level("trust ~ " + M2_TERMS, df_fit)
res3a = fit_3level("trust ~ " + M3A_TERMS, df_fit)
res3b = fit_3level("trust ~ " + M3B_TERMS, df_fit)

for name, res in (("M2", res2), ("M3a", res3a), ("M3b", res3b)):
    vc = variance_components(res)
    print(f"{name}: ICC={vc.icc_l3:.3f}  L2VPC={vc.vpc_l2:.3f}  loglik={res.llf:.1f}")

common-sample N (M2..M5): 165,969, 30 countries


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


M2: ICC=0.232  L2VPC=0.011  loglik=-180216.9
M3a: ICC=0.183  L2VPC=0.012  loglik=-180209.0
M3b: ICC=0.185  L2VPC=0.013  loglik=-180205.0


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


### Mundlak Wald: γ_W = γ_B?

In [4]:
for name, res in (("M3a (with round dummies)", res3a), ("M3b (no round dummies)", res3b)):
    test = mundlak_wald(res, "exposure_ct_within", "exposure_ct_between")
    print(f"--- {name}")
    print(f"  γ_W = {test['gamma_w']:+.4f}  γ_B = {test['gamma_b']:+.4f}")
    print(f"  diff = {test['diff']:+.4f}  SE = {test['se_diff']:.4f}")
    print(f"  Wald χ²(1) = {test['chi2']:.2f}  p = {test['pvalue']:.4f}")
    print()

--- M3a (with round dummies)
  γ_W = +0.0235  γ_B = +11.2390
  diff = -11.2155  SE = 3.6077
  Wald χ²(1) = 9.66  p = 0.0019

--- M3b (no round dummies)
  γ_W = +1.4591  γ_B = +11.1681
  diff = -9.7090  SE = 3.6266
  Wald χ²(1) = 7.17  p = 0.0074



## 3. M4 — random slope on `genai_z_gmc`

Group-mean centred so the random intercept variance σ²_u0 is interpretable at the country mean of exposure (Snijders & Bosker, lecture 3).

In [5]:
# Re-fit M3a with random slope on genai_z_gmc.
# Note: re_formula="~1 + genai_z_gmc" gives a 2x2 random-effect cov.
M4_TERMS = M3A_TERMS  # same fixed effects as M3a
res4 = fit_3level(
    "trust ~ " + M4_TERMS,
    df_fit,
    re_formula="~1 + genai_z_gmc",
)
vc4 = variance_components(res4)
print(f"M4 ICC={vc4.icc_l3:.3f}  L2VPC={vc4.vpc_l2:.3f}  loglik={res4.llf:.1f}")
print()
print("L3 random-effect covariance matrix (intercept, slope on genai_z_gmc):")
print(np.asarray(res4.cov_re).round(4))
if res4.cov_re.shape[0] >= 2:
    var_int = res4.cov_re.iloc[0, 0]
    var_slope = res4.cov_re.iloc[1, 1]
    cov_int_slope = res4.cov_re.iloc[0, 1]
    corr = cov_int_slope / np.sqrt(var_int * var_slope)
    print(f"\nimplied correlation(intercept, slope) = {corr:.3f}")

M4 ICC=0.097  L2VPC=0.011  loglik=-180065.0

L3 random-effect covariance matrix (intercept, slope on genai_z_gmc):
[[0.0552 0.0031]
 [0.0031 0.0127]]

implied correlation(intercept, slope) = 0.117


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


## 4. M5 — cross-level interaction `exposure_within × eisced_c`

Adjudicates H4a (substitution: more-educated react more strongly → negative interaction) vs H4b (curator: less strongly → positive) vs H4₀ (no moderation).

In [6]:
M5_TERMS = M4_TERMS + " + exposure_ct_within:eisced_c"
res5 = fit_3level(
    "trust ~ " + M5_TERMS,
    df_fit,
    re_formula="~1 + genai_z_gmc",
)
vc5 = variance_components(res5)
print(f"M5 ICC={vc5.icc_l3:.3f}  L2VPC={vc5.vpc_l2:.3f}  loglik={res5.llf:.1f}")
interaction_name = "exposure_ct_within:eisced_c"
if interaction_name in res5.params.index:
    beta = res5.params[interaction_name]
    se = res5.bse[interaction_name]
    z = beta / se
    from scipy.stats import norm
    p = 2 * (1 - norm.cdf(abs(z)))
    print(f"\ninteraction coefficient: β = {beta:+.4f}, SE = {se:.4f}, z = {z:+.2f}, p = {p:.3f}")
    if abs(z) > 1.96:
        if beta < 0:
            print("  → consistent with H4a (substitution): higher-educated react more strongly")
        else:
            print("  → consistent with H4b (curator): higher-educated react less strongly")
    else:
        print("  → consistent with H4₀ (no moderation by education)")

M5 ICC=0.185  L2VPC=0.012  loglik=-180042.8

interaction coefficient: β = -0.1422, SE = 0.1304, z = -1.09, p = 0.275
  → consistent with H4₀ (no moderation by education)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 5. M6 — institutional moderation by EPL_v1

Restrict to OECD countries with non-null `epl_c` (the L3 frame from Day 3 marks ~13 of 36 ESS countries with NaN EPL — they drop here, documented in §12).

In [7]:
df_m6 = df_fit.dropna(subset=["epl_c"]).copy()
# Centre EPL on its sample mean for interpretable main effect.
df_m6["epl_c_centred"] = df_m6["epl_c"] - df_m6["epl_c"].mean()
print(f"M6 sample: {len(df_m6):,} obs, {df_m6.cntry.nunique()} countries")

M6_TERMS = M5_TERMS + " + epl_c_centred + exposure_ct_within:epl_c_centred"
res6 = fit_3level(
    "trust ~ " + M6_TERMS,
    df_m6,
    re_formula="~1 + genai_z_gmc",
)
vc6 = variance_components(res6)
print(f"M6 ICC={vc6.icc_l3:.3f}  L2VPC={vc6.vpc_l2:.3f}  loglik={res6.llf:.1f}")
epl_int = "exposure_ct_within:epl_c_centred"
if epl_int in res6.params.index:
    print(f"\nEPL × within-exposure: β = {res6.params[epl_int]:+.4f}  SE = {res6.bse[epl_int]:.4f}")

M6 sample: 141,757 obs, 22 countries


M6 ICC=0.132  L2VPC=0.014  loglik=-151885.1

EPL × within-exposure: β = +1.6370  SE = 1.4425


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


## 6. Master table M0 → M6

Reload M0 and M1 from notebook 06 to get the full progression in one frame.

In [8]:
# Re-fit M0 and M1 on this notebook's common sample so all rows compare on the same N.
res0 = fit_3level("trust ~ 1", df_fit)
res1 = fit_3level("trust ~ " + M1_TERMS, df_fit)

results = {
    "M0":  res0,
    "M1":  res1,
    "M2":  res2,
    "M3a": res3a,
    "M3b": res3b,
    "M4":  res4,
    "M5":  res5,
    "M6":  res6,
}
tbl = master_table(
    results,
    coefs=(
        "genai_z",
        "exposure_ct_within",
        "exposure_ct_between",
        "exposure_ct_within:eisced_c",
        "epl_c_centred",
        "exposure_ct_within:epl_c_centred",
    ),
)
for c in tbl.select_dtypes("float64").columns:
    tbl[c] = tbl[c].round(4)
tbl

,model,n_obs,n_groups_l3,loglik,aic,bic,sigma_u0_sq,sigma_v0_sq,sigma_e_sq,icc_l3,...,exposure_ct_within__beta,exposure_ct_within__se,exposure_ct_between__beta,exposure_ct_between__se,exposure_ct_within:eisced_c__beta,exposure_ct_within:eisced_c__se,epl_c_centred__beta,epl_c_centred__se,exposure_ct_within:epl_c_centred__beta,exposure_ct_within:epl_c_centred__se
0,M0,165969,30,-184483.4813,NaN,NaN,0.1792,0.0181,0.5388,0.2435,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M1,165969,30,-180230.0389,NaN,NaN,0.1670,0.0148,0.5113,0.2410,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M2,165969,30,-180216.8833,NaN,NaN,0.1563,0.0073,0.5113,0.2316,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M3a,165969,30,-180208.9726,NaN,NaN,0.1165,0.0074,0.5113,0.1834,...,0.0235,1.1225,11.2390,3.4266,NaN,NaN,NaN,NaN,NaN,NaN
4,M3b,165969,30,-180205.0465,NaN,NaN,0.1181,0.0085,0.5113,0.1851,...,1.4591,1.1180,11.1681,3.4512,NaN,NaN,NaN,NaN,NaN,NaN
5,M4,165969,30,-180065.0088,NaN,NaN,0.0552,0.0063,0.5101,0.0966,...,0.0393,1.0443,11.1804,2.3732,NaN,NaN,NaN,NaN,NaN,NaN
6,M5,165969,30,-180042.7757,NaN,NaN,0.1178,0.0074,0.5101,0.1854,...,0.2035,1.1224,10.4884,3.0264,-0.1422,0.1304,NaN,NaN,NaN,NaN
7,M6,141757,22,-151885.0570,NaN,NaN,0.0767,0.0083,0.4965,0.1319,...,0.6977,1.3183,7.5423,4.0032,-0.3177,0.1414,-0.1072,0.0973,1.637,1.4425


## 7. Persist results

In [9]:
out_path = INTERIM_DIR / "master_table_m0_m6.parquet"
tbl.to_parquet(out_path, index=False)
print(f"wrote {out_path}")

# Save the Mundlak Wald summary for the results section.
wald_records = [
    {"model": name, **mundlak_wald(res, "exposure_ct_within", "exposure_ct_between")}
    for name, res in (("M3a", res3a), ("M3b", res3b), ("M4", res4), ("M5", res5))
    if "exposure_ct_within" in res.params.index
]
wald_df = pd.DataFrame(wald_records)
wald_df.to_parquet(INTERIM_DIR / "mundlak_wald_tests.parquet", index=False)
wald_df.round(4)

wrote /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/interim/master_table_m0_m6.parquet


,model,gamma_w,gamma_b,diff,se_diff,chi2,df,pvalue
0,M3a,0.0235,11.2390,-11.2155,3.6077,9.6643,1.0,0.0019
1,M3b,1.4591,11.1681,-9.7090,3.6266,7.1674,1.0,0.0074
2,M4,0.0393,11.1804,-11.1411,2.5961,18.4162,1.0,0.0000
3,M5,0.2035,10.4884,-10.2849,3.2335,10.1171,1.0,0.0015


## 8. Day-5 hard checkpoint (Track A)

Plan §11 Day 5: master regression table M0→M6 + robustness summary + Mundlak Wald χ². Track A here delivers the master table + Wald.

In [10]:
checks = {
    "M3a converged": res3a.converged,
    "M3b converged": res3b.converged,
    "M4  converged": res4.converged,
    "M5  converged": res5.converged,
    "M6  converged": res6.converged,
    "both within & between in M3a": all(
        c in res3a.params.index for c in ("exposure_ct_within", "exposure_ct_between")
    ),
    "interaction in M5": "exposure_ct_within:eisced_c" in res5.params.index,
    "EPL interaction in M6": "exposure_ct_within:epl_c_centred" in res6.params.index,
}
for k, v in checks.items():
    print(f"  {k}: {'PASS' if v else 'FAIL'}")
print()
print("Day-5 Track A:", "PASS" if all(checks.values()) else "pending")

  M3a converged: PASS
  M3b converged: PASS
  M4  converged: PASS
  M5  converged: PASS
  M6  converged: PASS
  both within & between in M3a: PASS
  interaction in M5: PASS
  EPL interaction in M6: PASS

Day-5 Track A: PASS
